<a href="https://colab.research.google.com/github/rafaelatorresbrenda-coder/Analise-LLM-Caderno-2021-1-1-AZUL-APLICACAO_DIGITAL/blob/main/CategorizacaoQuestoesENEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Projeto Categorização das Questões do ENEM
Etapa 2/5: Extração dos Conteúdos por Leitura de Imagens e OCR via IA

Objetivo: Processar as imagens de questões recortadas do ENEM (do passo 1) para extrair o texto fiel e descrever imagens/figuras usando o modelo Gemini 2.5.

Justificativa: esse notebook foi feito para mitigar as complexidades de instalação nos computadores dos colégios em que dou aula.

## O que é o Google Colab? E o que é um notebook?
O Google Colab é uma plataforma da Google, um recurso para rodar comandos em nuvem.

Em outras palavras, podemos dizer que a Google tem supercomputadores, e que agora estamos acessando esses supercomputadores pela internet para rodar comandos e armazenar arquivos.

Dessa forma, eliminamos as complexidades de instalação, versionamento e limitações dos computadores nos laboratórios físicos.

Um notebook é um arquivo onde podemos escrever texto e código para serem executados em nuvem.

É interessante um notebook porque podemos escrever texto para explicar os códigos, e os códigos para executá-los, tudo na mesma página.

##PASSO 1 - Enviar as imagens das questões para o Google Drive
- Abra seu Google Drive.
- Acesse a pasta "Colab Notebooks".
- Pegue a pasta das imagens finalizadas que você tratou no repositório de Tratamento de Imagens e faça o upload para a pasta "Colab Notebooks" do Google Drive.
- Volte para seu notebook do Google Colab e recarrege a página.
- Confira se sua pasta finalizadas está aparecendo ao acessar o menu lateral do notebook > drive > MyDrive > Colab Notebook. Deve aparecer a pasta "finalizadas".

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Passo 2: Baixando e configurando a API

###Instalar o SDK oficial do Google GenAI

In [10]:
!pip install -q google-genai
print("✅ SDK google-genai instalado com sucesso!")

✅ SDK google-genai instalado com sucesso!


### Criar e adicionar chave de API
1. No menu lateral esquerdo desta página, clique no ícone de "chave".
2. Clique em "Chaves da API Gemini", e então "Gerenciar Chaves no Google AI Studio"
3. Na nova aba, procure no canto superior direito por "Criar chave de API" e clique.
4. Preencha os campos para criar a nova chave. Escolha o nome ```GEMINI_API_KEY```, e escolha um projeto; se for preciso, crie um projeto chamado "enem".
5. Volte para os Secrets do seu notebook. Clique em "Chaves da API Gemini" e então "Importar chave do Google AI Studio".
6. Se não der certo a importação, adicione manualmente clicando em "+ Adicionar novo secret", ative a opção "Acesso ao notebook", dê o nome de ```GEMINI_API_KEY``` e cole no valor a sua "Chave de API".

## Passo 3: Execução do código

### Autenticação

In [11]:
import os
from google.colab import userdata

# Busca a chave configurada nos Secrets do Colab e define a variável de ambiente
try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("🔑 Chave GEMINI_API_KEY carregada com sucesso!")
except Exception as e:
    print("❌ Erro ao carregar a chave! Verifique se você salvou no ícone do Cadeado (Secrets).")

🔑 Chave GEMINI_API_KEY carregada com sucesso!


### Localização das imagens no Google Drive

In [6]:
import os
from google.colab import drive

# ==================== 1. MONTA O GOOGLE DRIVE ====================
print("📂 Conectando ao seu Google Drive...")
drive.mount('/content/drive')

# ==================== 2. DEFINE OS CAMINHOS ====================
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"

# Pasta com as imagens ORIGINAIS (.png) das questões
INPUT_DIR = os.path.join(BASE_DIR, "finalizadas")

# Pasta com os TXTs já extraídos (entrada da classificação)
TEXTOS_DIR = os.path.join(BASE_DIR, "textos_extraidos")

# Fontes normativas
ESTRUTURA_PATH = os.path.join(BASE_DIR, "estrutura.json")
GABARITO_PATH  = os.path.join(BASE_DIR, "Gabarito.txt")   # <-- atenção: .txt, não .json

# Pasta de SAÍDA da classificação (será criada se não existir)
OUTPUT_DIR = os.path.join(BASE_DIR, "conteudo_classificado")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==================== 3. VERIFICAÇÕES ====================
print("\n" + "=" * 60)
print("🔎 VERIFICAÇÃO DOS CAMINHOS")
print("=" * 60)

def checar_pasta(caminho, descricao, extensao=None):
    """Verifica existência de uma pasta e conta arquivos (opcionalmente por extensão)."""
    if not os.path.exists(caminho):
        print(f"❌ {descricao}: NÃO ENCONTRADA")
        print(f"   Caminho esperado: {caminho}")
        return 0
    arquivos = os.listdir(caminho)
    if extensao:
        arquivos = [f for f in arquivos if f.lower().endswith(extensao)]
    print(f"✅ {descricao}: OK")
    print(f"   Caminho: {caminho}")
    print(f"   Arquivos ({extensao or 'todos'}): {len(arquivos)}")
    return arquivos

def checar_arquivo(caminho, descricao):
    """Verifica existência de um arquivo específico."""
    if not os.path.exists(caminho):
        print(f"❌ {descricao}: NÃO ENCONTRADO")
        print(f"   Caminho esperado: {caminho}")
        return False
    tamanho_kb = os.path.getsize(caminho) / 1024
    print(f"✅ {descricao}: OK")
    print(f"   Caminho: {caminho}")
    print(f"   Tamanho: {tamanho_kb:.1f} KB")
    return True

# Pasta de imagens originais
imagens = checar_pasta(INPUT_DIR, "Pasta 'finalizadas' (imagens)", extensao=".png")
print()

# Pasta de TXTs extraídos
textos = checar_pasta(TEXTOS_DIR, "Pasta 'textos_extraidos' (TXTs)", extensao=".txt")
print()

# Arquivos normativos
tem_estrutura = checar_arquivo(ESTRUTURA_PATH, "estrutura.json")
print()
tem_gabarito  = checar_arquivo(GABARITO_PATH, "Gabarito.txt")
print()

# Pasta de saída
print(f"✅ Pasta de saída 'conteudo_classificado': pronta")
print(f"   Caminho: {OUTPUT_DIR}")
print()

# ==================== 4. RESUMO FINAL ====================
print("=" * 60)
print("📊 RESUMO")
print("=" * 60)
print(f"🖼️  Imagens .png em finalizadas : {len(imagens)}")
print(f"📝 TXTs em textos_extraidos         : {len(textos)}")
print(f"📚 estrutura.json                          : {'OK' if tem_estrutura else 'AUSENTE'}")
print(f"🎯 Gabarito.txt                            : {'OK' if tem_gabarito else 'AUSENTE'}")
print()

# Alerta: arquivos .txt na pasta de imagens (pode indicar bagunça)
extras = [f for f in os.listdir(INPUT_DIR)] if os.path.exists(INPUT_DIR) else []
nao_png = [f for f in extras if not f.lower().endswith(".png")]
if nao_png:
    print(f"⚠️  {len(nao_png)} arquivo(s) NÃO-.png em 'finalizadas':")
    for f in nao_png[:5]:
        print(f"    - {f}")
    if len(nao_png) > 5:
        print(f"    ... e mais {len(nao_png) - 5}.")
    print()

# Verifica se cada TXT tem gabarito correspondente (por nome)
if tem_gabarito and textos:
    with open(GABARITO_PATH, "r", encoding="utf-8") as f:
        conteudo_gab = f.read()
    faltando = [t for t in textos if os.path.splitext(t)[0] not in conteudo_gab]
    if faltando:
        print(f"⚠️  {len(faltando)} TXT(s) SEM gabarito correspondente no arquivo:")
        for f in faltando[:5]:
            print(f"    - {f}")
        if len(faltando) > 5:
            print(f"    ... e mais {len(faltando) - 5}.")
    else:
        print("✅ Todos os TXTs possuem gabarito correspondente.")
    print()

if len(imagens) > 0 and len(textos) == 0:
    print("💡 Dica: ainda não há TXTs em 'textos_extraidos_Maria'.")
    print("   Rode PRIMEIRO o script de extração (OCR + Gemini) para gerar os TXTs.")

print("\n🏁 Verificação concluída. Pronto para rodar a célula de processamento.")

📂 Conectando ao seu Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🔎 VERIFICAÇÃO DOS CAMINHOS
✅ Pasta 'finalizadas' (imagens): OK
   Caminho: /content/drive/MyDrive/Colab Notebooks/finalizadas
   Arquivos (.png): 90

✅ Pasta 'textos_extraidos' (TXTs): OK
   Caminho: /content/drive/MyDrive/Colab Notebooks/textos_extraidos
   Arquivos (.txt): 90

✅ estrutura.json: OK
   Caminho: /content/drive/MyDrive/Colab Notebooks/estrutura.json
   Tamanho: 1.6 KB

✅ Gabarito.txt: OK
   Caminho: /content/drive/MyDrive/Colab Notebooks/Gabarito.txt
   Tamanho: 0.6 KB

✅ Pasta de saída 'conteudo_classificado': pronta
   Caminho: /content/drive/MyDrive/Colab Notebooks/conteudo_classificado

📊 RESUMO
🖼️  Imagens .png em finalizadas : 90
📝 TXTs em textos_extraidos         : 90
📚 estrutura.json                          : OK
🎯 Gabarito.txt                            : OK

⚠️  90 TXT(s) SEM gabarito correspondent

### Executar o Processamento em Lote

In [7]:
import re
import json

def carregar_gabarito_txt(path):
    """
    Lê gabarito.txt e devolve um dict {id_questao: letra}.
    Aceita formatos comuns:
      - "2021_1_dia1_q45: C"
      - "2021_1_dia1_q45 - C"
      - "2021_1_dia1_q45 C"
      - "2021_1_dia1_q45=C"
      - "1 C"  /  "1: C"
    Ignora linhas vazias e comentários iniciados por '#'.
    """
    if not os.path.exists(path):
        return {}

    gabarito = {}
    with open(path, "r", encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue
            # Tenta separadores comuns
            match = re.match(r"^(.+?)\s*[:=\-–]\s*([A-Ea-e])\s*$", linha)
            if not match:
                # Última tentativa: último token é a letra
                match = re.match(r"^(.+?)\s+([A-Ea-e])$", linha)
            if match:
                chave = match.group(1).strip()
                letra = match.group(2).upper()
                gabarito[chave] = letra

    return gabarito


# Carrega e mostra amostra
gabarito_dict = carregar_gabarito_txt(GABARITO_PATH)
print(f"🎯 Gabarito carregado: {len(gabarito_dict)} entradas")
if gabarito_dict:
    amostra = list(gabarito_dict.items())[:5]
    print("   Amostra:")
    for k, v in amostra:
        print(f"     {k} → {v}")


🎯 Gabarito carregado: 90 entradas
   Amostra:
     ﻿1 → C
     2 → A
     3 → E
     4 → A
     5 → A


In [12]:
import os
import re
import json
import time
import copy
from datetime import datetime
from google import genai
from google.genai import types

# ==================== CONFIGURAÇÃO DE CAMINHOS ====================
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"

IMAGENS_DIR = os.path.join(BASE_DIR, "finalizadas")
INPUT_DIR   = os.path.join(BASE_DIR, "textos_extraidos")
OUTPUT_DIR  = os.path.join(BASE_DIR, "conteudo_classificado")

ESTRUTURA_PATH = os.path.join(BASE_DIR, "estrutura.json")
GABARITO_PATH  = os.path.join(BASE_DIR, "Gabarito.txt")
MATRIZ_PATH    = os.path.join(BASE_DIR, "matriz_referencia.json")

MODEL_NAME = "gemini-3.5-flash-lite"
MAX_RETRIES = 3
RETRY_DELAY = 5

# ==================== METADADOS FIXOS ====================
ESTUDANTE = {
    "nome_completo": "Brenda Rafaela Torres",
    "colegio": "Darcy José Costa",
    "turma": "3° B",
}

# 🆕 Valores fixos da identificação do caderno
# (aplicados a TODAS as questões, independente do nome do arquivo)
CADERNO_FIXO = {
    "ano": 2021,
    "dia": 1,
    "numero": None,          # preenchido por questão
    "cor": "Azul",
    "aplicacao": "Aplicação digital",
}

# ==================== PROMPT ====================
PROMPT = """
Você é um analista especialista em questões do ENEM e em classificação segundo a Matriz de Referência do ENEM.

FONTES E HIERARQUIA:
1. O TXT da questão é a fonte primária do conteúdo, comando e alternativas.
2. MATRIZ_REFERENCIA (fornecida no contexto) é a fonte normativa para área, competência, habilidades, eixo cognitivo e objetos/conteúdos da Matriz.
3. gabarito.txt é a fonte oficial do gabarito quando houver correspondência.

REGRAS ABSOLUTAS:
- Não invente informações. Se não houver evidência, use null ou [].
- Nunca crie área, competência, habilidade ou eixo fora da MATRIZ_REFERENCIA.
- Ao selecionar item da Matriz, copie ID e texto EXATAMENTE como estão na MATRIZ_REFERENCIA.
- Nunca parafraseie, resuma ou corrija a redação da Matriz.
- A habilidade principal representa principalmente o que o estudante precisa fazer para resolver a questão, não apenas o assunto superficial.
- Considere comando + operação cognitiva + conteúdo + alternativas + correspondência com a Matriz.
- Habilidades relacionadas só entram quando houver evidência clara.
- Gabarito oficial tem prioridade sobre qualquer inferência da LLM.
- Sem gabarito oficial, 'correta' deve permanecer null.
- Analise todas as alternativas presentes e não crie alternativas ausentes.
- Justificativas devem ser sustentadas pela questão.
- Não use conhecimento externo para preencher lacunas.
- OCR pode conter erros; corrija mentalmente apenas erros evidentes necessários à compreensão, sem inventar texto.
- Analise cada questão independentemente.
- Confiança: alta = correspondência clara; media = alguma ambiguidade; baixa = evidência insuficiente/ambígua.
- Retorne SOMENTE o JSON no schema fornecido, sem markdown, sem comentários.

A CLASSIFICAÇÃO DE CONTEÚDO é analítica e não deve ser apresentada como se fosse redação oficial da Matriz.
"""


# ==================== CARREGAMENTO DAS FONTES ====================
def carregar_json(path):
    if not os.path.exists(path):
        print(f"⚠️  Arquivo não encontrado: {path}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def carregar_gabarito_txt(path):
    """Lê gabarito.txt e devolve dict {numero_str: letra}."""
    if not os.path.exists(path):
        return {}
    gabarito = {}
    with open(path, "r", encoding="utf-8-sig") as f:
        for linha in f:
            linha = linha.strip()
            if not linha or linha.startswith("#"):
                continue
            match = re.match(r"^(\d+)\s*[:=\-–]\s*([A-Ea-e])\s*$", linha)
            if match:
                gabarito[str(int(match.group(1)))] = match.group(2).upper()
    return gabarito


def extrair_numero_do_arquivo(filename):
    """Extrai o número da questão do nome do arquivo (último número)."""
    nome = os.path.splitext(filename)[0]
    matches = re.findall(r"(?:q|questao|questão)?[_\s\-]*(\d+)", nome, re.IGNORECASE)
    if matches:
        return str(int(matches[-1]))
    return None


def buscar_gabarito(gabarito_dict, numero):
    if not gabarito_dict or numero is None:
        return None
    return gabarito_dict.get(str(numero))


# ==================== MERGE ESTRITO COM O MOLDE ====================
def merge_estrito(molde, dados_llm):
    """Reconstrói a saída seguindo RIGOROSAMENTE o molde do estrutura.json."""
    if isinstance(molde, dict):
        resultado = {}
        for chave, valor_molde in molde.items():
            if isinstance(dados_llm, dict) and chave in dados_llm:
                resultado[chave] = merge_estrito(valor_molde, dados_llm[chave])
            else:
                resultado[chave] = copy.deepcopy(valor_molde)
        return resultado

    if isinstance(molde, list):
        if isinstance(dados_llm, list):
            if len(molde) > 0 and len(dados_llm) > 0:
                return [merge_estrito(molde[0], item) for item in dados_llm]
            return copy.deepcopy(dados_llm)
        return copy.deepcopy(molde)

    if dados_llm is None:
        return copy.deepcopy(molde)
    if isinstance(molde, bool):
        return bool(dados_llm)
    if isinstance(molde, int) and not isinstance(molde, bool):
        try:
            return int(dados_llm)
        except (ValueError, TypeError):
            return molde
    return dados_llm


# ==================== CLIENTE GEMINI ====================
def inicializar_cliente():
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        print("❌ 'GEMINI_API_KEY' não definida nos Secrets do Colab.")
        return None
    try:
        client = genai.Client(api_key=api_key)
        print("🤖 Cliente Gemini pronto.")
        return client
    except Exception as e:
        print(f"❌ Erro ao inicializar Gemini: {e}")
        return None


def chamar_gemini(client, prompt, texto_questao, matriz, gab_oficial,
                  schema_exemplo, id_questao):
    contexto = f"""
================ MATRIZ DE REFERÊNCIA ENEM (fonte normativa) ================
{json.dumps(matriz, ensure_ascii=False)[:80000]}

================ GABARITO OFICIAL ================
{json.dumps(gab_oficial, ensure_ascii=False) if gab_oficial else "null"}

================ ID DA QUESTÃO ================
{id_questao}

================ SCHEMA DE SAÍDA OBRIGATÓRIO ================
{json.dumps(schema_exemplo, ensure_ascii=False, indent=2)}

================ TEXTO DA QUESTÃO (fonte primária) ================
{texto_questao}
=================================================================
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[prompt + "\n\n" + contexto],
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json",
        ),
    )
    return response.text


def parse_json_seguro(texto):
    if not texto:
        return None
    texto = texto.strip()
    if texto.startswith("```"):
        texto = texto.strip("`")
        if texto.lower().startswith("json"):
            texto = texto[4:]
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        ini, fim = texto.find("{"), texto.rfind("}")
        if ini != -1 and fim > ini:
            try:
                return json.loads(texto[ini:fim + 1])
            except json.JSONDecodeError:
                return None
    return None


# ==================== COMPLEMENTAÇÃO PÓS-LLM ====================
def preencher_campos_automaticos(dados, filename, numero, gab_oficial):
    """
    Preenche campos do estrutura.json que NÃO vêm da LLM:
    - identificacao_caderno: FIXO (exceto numero, que varia por questão)
    - conteudo_questao: caminhos + gabarito
    - auditoria_da_ia: modelo, versão, data, proveniência
    - estudante: fixo
    """
    q = dados.setdefault("questao", {})
    cad = q.setdefault("identificacao_caderno", {})
    ident = q.setdefault("identificacao_questao", {})
    cont = q.setdefault("conteudo_questao", {})

    # 🆕 Valores FIXOS de identificacao_caderno (sobrescreve tudo)
    cad["ano"]       = CADERNO_FIXO["ano"]
    cad["dia"]       = CADERNO_FIXO["dia"]
    cad["cor"]       = CADERNO_FIXO["cor"]
    cad["aplicacao"] = CADERNO_FIXO["aplicacao"]
    # numero continua variando por questão
    if numero is not None:
        cad["numero"] = int(numero)
        ident["numero"] = int(numero)

    if "espanhol" in os.path.splitext(filename)[0].lower():
        ident["espanhol"] = True
    if "ingles" in os.path.splitext(filename)[0].lower() or \
       "inglês" in os.path.splitext(filename)[0].lower():
        ident["ingles"] = True

    # --- Caminhos e gabarito ---
    cont["caminho_texto"] = os.path.join(INPUT_DIR, filename).replace(BASE_DIR, "")
    cont["link_imagem"]   = os.path.join(IMAGENS_DIR,
                                          os.path.splitext(filename)[0] + ".png"
                                          ).replace(BASE_DIR, "")
    if gab_oficial:
        cont["gabarito"] = gab_oficial

    # --- Auditoria ---
    aud = dados.setdefault("auditoria_da_ia", {})
    aud["modelo"] = MODEL_NAME
    aud["versao"] = "1.0"
    aud["datetime"] = datetime.now().isoformat(timespec="seconds")
    prov = aud.setdefault("proveniencia", {})
    prov["arquivo_questao"] = filename
    prov["arquivo_matriz"]  = os.path.basename(MATRIZ_PATH) if os.path.exists(MATRIZ_PATH) else "MATRIZ_ENEM_EMBUTIDA"
    val = aud.setdefault("validacao_humana", {})
    val.setdefault("validado", False)
    val.setdefault("data_validacao", None)

    # --- Estudante ---
    dados["estudante"] = ESTUDANTE

    return dados


# ==================== PROCESSAMENTO ====================
def processar_pasta():
    client = inicializar_cliente()
    if client is None:
        return

    schema_exemplo = carregar_json(ESTRUTURA_PATH)
    if schema_exemplo is None:
        print("❌ estrutura.json (schema de saída) é obrigatório.")
        return

    matriz = carregar_json(MATRIZ_PATH)
    if matriz is None:
        print(f"⚠️  {MATRIZ_PATH} não encontrado. Usando conhecimento do modelo.")
        matriz = {}

    gabarito = carregar_gabarito_txt(GABARITO_PATH)
    print(f"📚 Schema carregado. Matriz: {'OK' if matriz else 'vazia'}. "
          f"Gabarito: {len(gabarito)} entradas.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    arquivos = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".txt")])
    if not arquivos:
        print(f"⚠️  Nenhum .txt em: {INPUT_DIR}")
        return
    print(f"\n🚀 Processando {len(arquivos)} arquivo(s)...\n")

    for filename in arquivos:
        input_path  = os.path.join(INPUT_DIR, filename)
        output_path = os.path.join(OUTPUT_DIR,
                                   os.path.splitext(filename)[0] + ".json")

        if os.path.exists(output_path):
            print(f"⏭️  Já processado: {filename}")
            continue

        numero = extrair_numero_do_arquivo(filename)
        gab_oficial = buscar_gabarito(gabarito, numero)
        print(f"-> {filename}  |  nº={numero}  |  gabarito={gab_oficial}")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                texto_questao = f.read()

            resposta_raw = None
            for tentativa in range(1, MAX_RETRIES + 1):
                try:
                    resposta_raw = chamar_gemini(
                        client, PROMPT, texto_questao, matriz, gab_oficial,
                        schema_exemplo, numero
                    )
                    break
                except Exception as e:
                    print(f"   ⚠️  Tentativa {tentativa}/{MAX_RETRIES}: {e}")
                    if tentativa < MAX_RETRIES:
                        time.sleep(RETRY_DELAY)

            if resposta_raw is None:
                raise RuntimeError("Falha após todas as tentativas")

            dados_llm = parse_json_seguro(resposta_raw)
            if dados_llm is None:
                raise RuntimeError("Resposta não é JSON válido")

            # 🔒 MERGE ESTRITO: saída com chaves EXATAS do molde
            dados = merge_estrito(schema_exemplo, dados_llm)

            # Força boolean em 'correta'
            if gab_oficial:
                for alt in dados.get("analise_alternativas") or []:
                    if isinstance(alt, dict):
                        alt["correta"] = (
                            str(alt.get("letra", "")).upper() == gab_oficial
                        )

            # Preenche campos automáticos
            dados = preencher_campos_automaticos(
                dados, filename, numero, gab_oficial
            )

            # Sanidade: chaves de topo batem com o molde
            assert set(dados.keys()) == set(schema_exemplo.keys()), \
                "Chaves de topo divergem do molde!"

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(dados, f, ensure_ascii=False, indent=2)

            print(f"   ✅ Salvo: conteudo_classificado/{os.path.basename(output_path)}")

        except Exception as e:
            print(f"   ❌ FALHA em {filename}: {e}")
            erro_doc = copy.deepcopy(schema_exemplo)
            erro_doc["_erro"] = str(e)
            erro_doc["_arquivo_origem"] = filename
            erro_doc["_datetime"] = datetime.now().isoformat()
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(erro_doc, f, ensure_ascii=False, indent=2)

    print("\n🏁 Processamento concluído.")


# ==================== EXECUTA ====================
processar_pasta()

🤖 Cliente Gemini pronto.
⚠️  Arquivo não encontrado: /content/drive/MyDrive/Colab Notebooks/matriz_referencia.json
⚠️  /content/drive/MyDrive/Colab Notebooks/matriz_referencia.json não encontrado. Usando conhecimento do modelo.
📚 Schema carregado. Matriz: vazia. Gabarito: 90 entradas.

🚀 Processando 90 arquivo(s)...

⏭️  Já processado: questao-1-espanhol.txt
⏭️  Já processado: questao-10.txt
⏭️  Já processado: questao-11.txt
⏭️  Já processado: questao-12.txt
⏭️  Já processado: questao-13.txt
⏭️  Já processado: questao-14.txt
⏭️  Já processado: questao-15.txt
⏭️  Já processado: questao-16.txt
⏭️  Já processado: questao-17.txt
⏭️  Já processado: questao-18.txt
⏭️  Já processado: questao-19.txt
⏭️  Já processado: questao-2-espanhol.txt
⏭️  Já processado: questao-20.txt
⏭️  Já processado: questao-21.txt
⏭️  Já processado: questao-22.txt
⏭️  Já processado: questao-23.txt
⏭️  Já processado: questao-24.txt
⏭️  Já processado: questao-25.txt
⏭️  Já processado: questao-26.txt
⏭️  Já processado: 